# 04 — Multi-SKU Inventory Simulation
Run reorder point optimization and inventory simulation across the top 5 SKUs.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.inventory_simulator import calculate_reorder_point, simulate_inventory

df = pd.read_csv('../data/processed/processed_sales.csv')
df['date'] = pd.to_datetime(df['date'])
print('Loaded:', df.shape)

In [ ]:
# Top 5 SKUs by total sales volume
top_items = (df.groupby('item')['sales']
               .sum()
               .sort_values(ascending=False)
               .head(5)
               .index.tolist())

print('Top 5 items:', top_items)

In [ ]:
results = {}

for item in top_items:
    sku_df = (df[df['item'] == item]
              .groupby('date', as_index=False)['sales']
              .sum()
              .sort_values('date'))

    train = sku_df.iloc[:-90].copy()
    test  = sku_df.iloc[-90:].copy().reset_index(drop=True)

    reorder_point, safety_stock = calculate_reorder_point(train, lead_time=7, service_level=0.95)
    order_qty = int(train['sales'].mean() * 7)

    result = simulate_inventory(
        test,
        reorder_point=reorder_point,
        order_quantity=order_qty,
        initial_inventory=200,
        lead_time=7,
        holding_cost_per_unit=1,
        stockout_cost_per_unit=5,
    )

    results[item] = {**result, 'reorder_point': reorder_point,
                     'safety_stock': safety_stock, 'order_qty': order_qty, 'test_df': test}

    print(f'Item {item:>2} | ROP={reorder_point:>6.1f} | SS={safety_stock:>5.1f} | '
          f'Stockouts={result["stockouts"]:>5.0f} | Total Cost={result["total_cost"]:>9,.0f}')

## Summary Table

In [ ]:
summary = pd.DataFrame([{
    'Item':          item,
    'Reorder Point': round(results[item]['reorder_point'], 1),
    'Safety Stock':  round(results[item]['safety_stock'], 1),
    'Order Qty':     results[item]['order_qty'],
    'Stockouts':     round(results[item]['stockouts']),
    'Holding Cost':  results[item]['holding_cost'],
    'Stockout Cost': results[item]['stockout_cost'],
    'Total Cost':    results[item]['total_cost'],
} for item in top_items])

summary

## Inventory Level Charts

In [ ]:
fig, axes = plt.subplots(len(top_items), 1, figsize=(12, 4 * len(top_items)))

for ax, item in zip(axes, top_items):
    r = results[item]
    ax.plot(r['test_df']['date'].values, r['inventory_history'], label='Inventory')
    ax.axhline(r['reorder_point'], color='red',    linestyle='--', label=f"ROP ({r['reorder_point']:.0f})")
    ax.axhline(r['safety_stock'],  color='orange', linestyle=':',  label=f"Safety Stock ({r['safety_stock']:.0f})")
    ax.set_title(f'Item {item} — 90-Day Inventory Simulation')
    ax.set_ylabel('Units')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()